In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gdt.core.data_primitives import TimeBins
from bctools.analysis import BayesianBlocksLightcurve
import os

base_path = "/data/grb_eliza/sources/batch-1/flux-10-100/source_files"
file_name = "batch1_lc_dataset.npz"

data = np.load(base_path+"/"+file_name, allow_pickle=True)
grb = data["grb"]
tstart = data["tstart"]
duration = data["duration"]
lc = data["lc"]


In [ ]:



def analyze_lc(lightcurve, p0=0.05, isRate=False, panels=['z0', 'z1', 'x0', 'x1', 'y0', 'y1']):
    """
    Analyze the light curve data.
    Input:
        - lightcurve: data file with times and counts
        - p0: false alarm probability for the Bayesian Block algorithm
        - isRate: if True, the lightcurve files contains rates instead of counts
        - panels: list of detectors for which we have lightcurves
    """

    data = lightcurve  
    time = data[:,0,0] 
    time = time - time[0]

    signal = {}
    for i, panel in enumerate(panels):
        signal[panel] = data[:, i, 1]
        if isRate:
            signal[panel] = signal[panel] * bin_width

    
    # Construct light curve object
    bin_width = time[1] - time[0]
    bin_edges = np.zeros(len(time) + 1)
    bin_edges[1:-1] = (time[:-1] + time[1:]) / 2
    bin_edges[0] = time[0] - (time[1] - time[0]) / 2
    bin_edges[-1] = time[-1] + (time[-1] - time[-2]) / 2
    lo_edges, hi_edges = bin_edges[:-1], bin_edges[1:]
    exposure = np.full(len(time), bin_width)


    lc = {} # lc per panel
    for panel in panels:
        signal_panel = signal[panel]
        lc[panel] = TimeBins(signal_panel, lo_edges, hi_edges, exposure)
    #if len(panels) > 1: lc_psum = TimeBins.sum([lcs for lcs in lc.values()])
    #else: lc_psum = lc[panels[0]]
    lc_psum = lc[panels[0]]

    lc_sel = lc_psum
    
    fig1, ax1 = plt.subplots(figsize=(10,4))

    ax1.step(lc_sel.centroids, lc_sel.counts, where="mid")

    ax1.set_xlabel("Time [s]")
    ax1.set_ylabel("Counts / bin")
    ax1.set_title("Light curve")
    ax1.grid(True, alpha=0.3)

    # Apply Bayesian Blocks algorithm

    try: 
        bb_lc = BayesianBlocksLightcurve(lc_sel)
        
        bb_lc.compute_bayesian_blocks(p0=p0)
        
        signal_range = bb_lc.signal_range
        
        t90 = bb_lc.duration(quantile = .9)
        
        t90_error = bb_lc.duration_error(.9, nsamples = 100)
        
    except Exception as e:
        print(e)
        print('WARNING: ')
        return lc_sel, None, -9999, -9999, -9999, -9999, -9999, -9999, -9999, None, None
    fig2,ax2 = plt.subplots()
    ax2 = bb_lc.plot(ax=ax2)
  

    # Li&Ma calculation of the significance
    signal_lc = lc_sel.slice(bb_lc.signal_range.tstart, bb_lc.signal_range.tstop)
    bkg_lc = lc_sel.slice(bb_lc.signal_range.tstop, lc_sel.centroids[-1])
    bkg_lc = lc_sel.slice(lc_sel.centroids[0], bb_lc.signal_range.tstart)
    t_on = np.sum(signal_lc.exposure)
    t_off = np.sum(bkg_lc.exposure)
    alpha = t_on / t_off
    N_on = np.sum(signal_lc.rates * signal_lc.exposure)
    N_off = np.sum(bkg_lc.rates * bkg_lc.exposure)
    S = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5

    # Li&Ma calculation of the peak significance
    significance = []
    for rate, exp in zip(signal_lc.rates, signal_lc.exposure):
        N_on = rate * exp
        t_on = exp
        alpha = t_on / t_off
        S_bin = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5
        significance.append(S_bin)
    significance = np.array(significance)
    S_peak = np.max(significance)

    result = (lc_sel, bb_lc, signal_range.tstart, signal_range.tstop, t90, t90_error[0], t90_error[1], S, S_peak, ax1,ax2,lo_edges,hi_edges)
    plt.close(fig1)
    plt.close(fig2)

    return result


In [ ]:
index = 1
print("GRB: "+str(index)+" "+grb[index])
output = analyze_lc(lc[index], p0=0.002, isRate=False, panels=['z1', 'z0', 'x1', 'x0', 'y1', 'y0'])
fig1, ax1 = output[9]
fig2, ax2 = output[10]

# e lo mostri quando vuoi
plt.show()
print("duration"+str(output[4])+"\n\n")

In [ ]:
for i in range(0,50):
    print("GRB: "+str(i)+" "+grb[i])
    output = analyze_lc(lc[i], p0=0.01, isRate=False, panels=['z1', 'z0', 'x1', 'x0', 'y1', 'y0'])
    print("duration"+str(output[4])+"\n\n")